# **Причинно-следственный анализ: влияние кризиса на акции немецких автомобильных компаний**
## Дизельгейт (2015) и COVID-19 (2020) - CausalImpact + OLS Synthetic Control
**Автор:** Дубинкин Владислав

**Стек:** Python, CausalImpact (BSTS), OLS Counterfactual, Bootstrap CI

**Данные:** Ежедневные цены акций VW, BMW, Mercedes-Benz за 2010-2025 годы (примерно 4060 торговых дней/компанию)

In [ ]:
# Сетап
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

install("causalimpact")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")

# Для воспроизводимости
SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "figure.dpi":      130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size":       11,
})

In [ ]:
# Загружаем src-модули напрямую из репозитория
import urllib.request, importlib, sys

BASE = (
    "https://raw.githubusercontent.com/VladislavDubinkin/"
    "Causal-Inference-Automotive/main/src/"
)

for module in ["data_loader", "causal_impact_wrapper", "synthetic_control"]:
    url  = BASE + module + ".py"
    path = f"/tmp/{module}.py"
    urllib.request.urlretrieve(url, path)

if "/tmp" not in sys.path:
    sys.path.insert(0, "/tmp")

import data_loader          as dl
import causal_impact_wrapper as ciw
import synthetic_control     as sc

In [ ]:
# Загружаем данные
df = dl.load_raw()

print(f"Датасет: {df.shape[0]:,} строк × {df.shape[1]} столбцов")
print(f"Период:  {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Компании: {sorted(df['Company'].unique())}")
print(f"\nNaN в log_return: {df['log_return'].isna().sum()}")
df.head(3)

## 1. Разведочный анализ данных (EDA)

In [ ]:
vol   = dl.compute_volatility(df)
prices_norm = dl.compute_prices_normalized(df)
events = dl.get_event_windows()

print("DataFrame волатильности (широкий формат):")
print(vol.describe().round(4))

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=False)
colors = {"VW": "#1f77b4", "BMW": "#2ca02c", "MB": "#d62728", "DAX": "grey"}

# Цены
for ticker in ["VW", "BMW", "MB", "DAX"]:
    ax1.plot(prices_norm.index, prices_norm[ticker],
             label=ticker, color=colors[ticker], lw=1.2, alpha=0.85)

for ev in events.values():
    ax1.axvspan(pd.Timestamp(ev["pre"][1]),  pd.Timestamp(ev["post"][1]),
                alpha=0.08, color="orange" if "diesel" in ev["label"].lower() else "red")
    ax1.axvline(pd.Timestamp(ev["post"][0]), color="red", ls=":", lw=1.5)

ax1.set_title("Акции немецких автопроизводителей — Нормализованная цена (100 = Янв 2010)",
              fontweight="bold")
ax1.set_ylabel("Нормализованная цена")
ax1.legend(); ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# Волатильность
for ticker in ["VW", "BMW", "MB"]:
    ax2.plot(vol.index, vol[ticker], label=ticker,
             color=colors[ticker], lw=1.0, alpha=0.8)

for ev in events.values():
    ax2.axvspan(pd.Timestamp(ev["post"][0]), pd.Timestamp(ev["post"][1]),
                alpha=0.1, color="orange" if "diesel" in ev["label"].lower() else "red")

ax2.set_title("Скользящая 21-дневная годовая волатильность", fontweight="bold")
ax2.set_ylabel("Годовая волатильность"); ax2.legend()
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.savefig("figures/eda_prices_vol.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2. Гипотезы и дизайн эксперимента

| # | Гипотеза | Метод |
|---|-----------|--------|
| H₁: | Дизельгейт вызвал **значительный** всплеск волатильности **только у VW** | CausalImpact (BSTS) |
| H₂: | COVID-19 вызвал **синхронный** шок у **всех трех** компаний | OLS Synthetic Control |
| H₃: | Величина шока COVID-19 **превышает** Дизельгейт для всех компаний | Сравнительная таблица |

**Стратегия причинной идентификации:**  
Оба события являются экзогенными шоками с известными датами → пригодны в качестве естественных экспериментов.  
1) Дизельгейт: VW - объект воздействия; BMW + MB + DAX служат контрольными ковариатами  
2) COVID-19: все три компании — объекты воздействия; DAX (широкий рынок) — донор

---
## 3. CausalImpact — Дизельгейт (сентябрь 2015)

In [ ]:
ev = events["dieselgate"]

# Добавляем нормализованный DAX (ковариата волатильности)
dax_vol = df.drop_duplicates("Date").set_index("Date")["DAX"].sort_index()
dax_vol_roll = dax_vol.pct_change().rolling(21).std() * np.sqrt(252)
vol["DAX_vol"] = dax_vol_roll

ci_vw = ciw.run_causal_impact(
    vol_wide   = vol,
    target     = "VW",
    controls   = ["BMW", "MB", "DAX_vol"],
    pre_period = ev["pre"],
    post_period= ev["post"],
)

print(ci_vw.summary())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=False)

ciw.plot_causal_impact(
    ci         = ci_vw,
    event_date = ev["post"][0],
    title      = "VW — Дизельгейт",
    ax_triplet = axes,
)

plt.tight_layout()
plt.savefig("figures/causalimpact_dieselgate_vw.png", dpi=150, bbox_inches="tight")
plt.show()

metrics_diesel = ciw.extract_summary_metrics(ci_vw, "VW — Дизельгейт")
print(metrics_diesel)

In [ ]:
real_effect, placebo_effects = ciw.placebo_test(
    vol_wide    = vol,
    true_target = "VW",
    controls    = ["BMW", "MB"],
    pre_period  = ev["pre"],
    post_period = ev["post"],
)

p_placebo = np.mean(np.abs(placebo_effects) >= np.abs(real_effect))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(placebo_effects, bins=15, color="lightgrey", edgecolor="black")
ax.axvline(real_effect, color="red", lw=2.5,
           label=f"Реальный эффект VW = {real_effect:.3f}\nP-значение плацебо = {p_placebo:.3f}")
ax.set_title("Тест на перестановку плацебо — Дизельгейт", fontweight="bold")
ax.set_xlabel("Средний размер эффекта"); ax.legend()
plt.tight_layout()
plt.savefig("figures/placebo_dieselgate.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nРеальный эффект:    {real_effect:.4f}")
print(f"P-значение плацебо: {p_placebo:.3f}")

---
## 4. OLS Synthetic Control — COVID-19 (февраль 2020)
**Подход:** подгонка OLS на доковидном периоде (цель ~ DAX), экстраполяция контрфактической модели.  
Bootstrap CI (n=500) количественно оценивает неопределенность. Разрыв = наблюдаемое − контрфактическое.

In [ ]:
ev_covid = events["covid"]
results_covid = {}

fig, axes = plt.subplots(3, 1, figsize=(13, 12), sharex=False)

for i, ticker in enumerate(["VW", "BMW", "MB"]):
    model, cf = sc.fit_ols_counterfactual(
        prices_norm = prices_norm,
        target      = ticker,
        donor       = "DAX",
        pre_end     = ev_covid["pre"][1],
    )

    cf_mean, cf_lower, cf_upper = sc.bootstrap_ci(
        prices_norm = prices_norm,
        target      = ticker,
        donor       = "DAX",
        pre_end     = ev_covid["pre"][1],
        post_start  = ev_covid["post"][0],
        n_boot      = 500,
        seed        = SEED,
    )

    gap = sc.compute_gap_effect(prices_norm, ticker, cf, ev_covid["post"][0])
    results_covid[ticker] = gap

    sc.plot_synthetic_control(
        prices_norm   = prices_norm,
        target        = ticker,
        counterfactual= cf,
        lower_ci      = cf_lower,
        upper_ci      = cf_upper,
        event_date    = ev_covid["post"][0],
        title         = f"{ticker} — Синтетический контроль COVID-19",
        ax            = axes[i],
    )

plt.tight_layout()
plt.savefig("figures/synthetic_control_covid.png", dpi=150, bbox_inches="tight")
plt.show()

pd.DataFrame(results_covid).T

---
## 5. Сравнительные результаты

In [ ]:
rows = [
    {**ciw.extract_summary_metrics(ci_vw, "VW — Дизельгейт"),
     "event": "Дизельгейт", "company": "VW", "method": "CausalImpact"},
]

# CausalImpact для BMW и MB при Дизельгейте (проверка на распространение)
for ticker in ["BMW", "MB"]:
    controls_other = [c for c in ["VW", "BMW", "MB", "DAX_vol"] if c != ticker]
    ci_tmp = ciw.run_causal_impact(vol, ticker, controls_other,
                                   events["dieselgate"]["pre"],
                                   events["dieselgate"]["post"])
    rows.append({
        **ciw.extract_summary_metrics(ci_tmp, f"{ticker} — Дизельгейт"),
        "event": "Дизельгейт", "company": ticker, "method": "CausalImpact",
    })

# COVID из results_covid
for ticker, gap in results_covid.items():
    rows.append({
        "label":          f"{ticker} — COVID-19",
        "event":          "COVID-19",
        "company":        ticker,
        "method":         "OLS Synthetic Control",
        "avg_gap":        gap["avg_gap"],
        "rel_effect_pct": gap["rel_effect_pct"],
        "p_value":        "bootstrap",
    })

summary_df = pd.DataFrame(rows)[
    ["event", "company", "method", "rel_effect_pct", "avg_gap", "p_value"]
].rename(columns={
    "event": "Событие",
    "company": "Компания",
    "method": "Метод",
    "rel_effect_pct": "Относ. эффект (%)",
    "avg_gap":        "Средний разрыв",
    "p_value":        "p-значение",
})

print(summary_df.to_markdown(index=False))

---
## 6. Выводы для управления рисками

- **Идиосинкратический шок (Дизельгейт):** значим только для VW $\implies$ для портфелей  
  с экспозицией на VW нужен одиночный хедж, специфичный для акции, а не секторальный ETF.  
- **Системный шок (COVID-19):** синхронный шок у всех трёх $\implies$ стресс-сценарий для всего сектора;  
  стандартный исторический VaR занижает хвостовой риск примерно в 2 раза.  
- **Стресс-тестирование:** эффект CausalImpact от Дизельгейта можно  
  использовать как калиброванный сценарий для регуляторного стресс-теста.

## 7. Ограничения

1. **Нарушение SUTVA:** BMW/MB использовались как контроли для VW, но сами  
   частично пострадали от заражения $\implies$ контрфактическая модель может быть занижена.  
2. **Одиночный донор (COVID):** OLS с одним донором (DAX) менее "стабильный",  
   чем полноценный синтетический контроль с пулом из 10+ стран.  
3. **Параллельные тренды:** ключевое допущение — можно проверить только косвенно  
   через качество подгонки допериода (R² модели).
